# IBM Kingston Hardware Characterization and Region Recommendation

This notebook analyzes the supplied calibration snapshot and coupling topology, then recommends one connected 16-qubit region and one connected 32-qubit region. It is designed to run top-to-bottom with only the Python standard library.

**Decision rule.** Prefer regions with low readout and `sx` error, long coherence times, and low internal CZ error. A recommended region must be connected in the supplied undirected coupling map; unavailable calibration values are not treated as zero.


## 1. Load and validate the supplied files

Place this notebook in the same directory as the calibration file and topology JSON, or set `DATA_DIR` below. The calibration download is CSV-formatted despite its `.xls` filename, so it is read with `csv.DictReader`.


In [2]:
from pathlib import Path
import csv, json, math, statistics

DATA_DIR = Path.cwd()
calibration_candidates = sorted(DATA_DIR.glob("ibm_kingston_calibrations*"))
topology_candidates = sorted(DATA_DIR.glob("ibm_kingston_topology_complete*.json"))
if not calibration_candidates or not topology_candidates:
    raise FileNotFoundError("Put both supplied files beside the notebook, or change DATA_DIR.")

CALIBRATION_FILE = calibration_candidates[0]
TOPOLOGY_FILE = topology_candidates[0]
rows = list(csv.DictReader(CALIBRATION_FILE.open(newline="")))
topology = json.load(TOPOLOGY_FILE.open())

print(f"Calibration rows: {len(rows)}; columns: {len(rows[0])}")
print(f"Topology: {topology['num_qubits']} qubits, {len(topology['coupling_map_undirected'])} undirected couplers")
print("All operational flags:", sorted({row['Operational'] for row in rows}))


Calibration rows: 156; columns: 19
Topology: 156 qubits, 176 undirected couplers
All operational flags: ['Yes']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Initial dataset analysis

The snapshot provides per-qubit T1/T2, readout error, single-qubit error, and neighbor-specific CZ error. The topology is a single connected 156-qubit graph with 176 undirected couplers. Calibration completeness is explicitly checked because missing values should not receive an artificial quality benefit.


In [3]:
numeric_columns = ["T1 (us)", "T2 (us)", "Readout assignment error", "√x (sx) error", "CZ error"]
for column in numeric_columns:
    missing = [int(row["Qubit"]) for row in rows if not row[column]]
    print(f"{column}: missing on {missing if missing else 'none'}")

for column in ["T1 (us)", "T2 (us)", "Readout assignment error", "√x (sx) error"]:
    values = [float(row[column]) for row in rows if row[column]]
    print(f"{column:27s} min={min(values):.6g}, median={statistics.median(values):.6g}, mean={statistics.mean(values):.6g}, max={max(values):.6g}")

missing_required = [int(row["Qubit"]) for row in rows
                    if not (row["T1 (us)"] and row["T2 (us)"] and row["√x (sx) error"])]
print("Excluded from ranking because a required calibration is missing:", missing_required)


T1 (us): missing on [146]
T2 (us): missing on [146]
Readout assignment error: missing on none
√x (sx) error: missing on [112, 113, 146]
CZ error: missing on [112, 120, 146]
T1 (us)                     min=65.8875, median=280.249, mean=271.047, max=479.73
T2 (us)                     min=11.9266, median=134.261, mean=171.809, max=488.482
Readout assignment error    min=0.00317383, median=0.00927734, mean=0.0246347, max=0.504883
√x (sx) error               min=8.66076e-05, median=0.000214317, mean=0.000301885, max=0.00583784
Excluded from ranking because a required calibration is missing: [112, 113, 146]


## 3. Hardware representation and scoring methodology

I represent the device as an undirected graph: qubits are vertices and permitted two-qubit interactions are edges. For each eligible qubit, three percentile-ranked quantities are combined:

- 45% low readout assignment error;
- 25% low `sx` error;
- 30% high geometric-mean coherence, `sqrt(T1*T2)`.

For each physical coupler, the two directed CZ-error reports are averaged and percentile-ranked with lower error preferred. A candidate region's score is `0.50 × mean node quality + 0.40 × mean internal-coupler quality + 0.10 × compactness`, where compactness rewards useful internal connectivity. Percentile ranks make the differently-scaled measurements comparable while preserving their ordering. These weights are a transparent engineering preference, not a claim that they are the only valid workload model.


In [4]:
def parse_pairs(value):
    """Parse fields such as '1:0.001;3:0.002' into {1: 0.001, 3: 0.002}."""
    if not value:
        return {}
    return {int(k): float(v) for item in value.split(";") for k, v in [item.split(":")]}

def rank01(values):
    """Percentile rank in [0, 1], with larger values always better."""
    ordered = sorted(values.items(), key=lambda pair: pair[1])
    return {key: rank / (len(ordered) - 1) for rank, (key, _) in enumerate(ordered)}

eligible_rows = [row for row in rows if row["T1 (us)"] and row["T2 (us)"] and row["√x (sx) error"]]
by_qubit = {int(row["Qubit"]): row for row in eligible_rows}

readout_rank = rank01({q: -float(row["Readout assignment error"]) for q, row in by_qubit.items()})
sx_rank = rank01({q: -float(row["√x (sx) error"]) for q, row in by_qubit.items()})
coherence_rank = rank01({q: math.sqrt(float(row["T1 (us)"]) * float(row["T2 (us)"]))
                         for q, row in by_qubit.items()})
node_quality = {q: 0.45 * readout_rank[q] + 0.25 * sx_rank[q] + 0.30 * coherence_rank[q]
                for q in by_qubit}

directed_cz = {(q, neighbor): error
               for q, row in by_qubit.items()
               for neighbor, error in parse_pairs(row["CZ error"]).items()}
adjacency = {q: [] for q in by_qubit}
raw_edge_error = {}
for u, v in topology["coupling_map_undirected"]:
    if u not in by_qubit or v not in by_qubit:
        continue
    reports = [directed_cz[pair] for pair in ((u, v), (v, u)) if pair in directed_cz]
    if len(reports) != 2:
        continue                         # Do not infer a missing CZ calibration.
    adjacency[u].append(v)
    adjacency[v].append(u)
    raw_edge_error[(u, v)] = statistics.mean(reports)

edge_quality = rank01({edge: -error for edge, error in raw_edge_error.items()})
print(f"Eligible qubits: {len(by_qubit)}; eligible couplers with two CZ reports: {len(raw_edge_error)}")


Eligible qubits: 153; eligible couplers with two CZ reports: 168


## 4. Connected-region search

Finding the globally optimum connected subgraph under a multi-metric objective is combinatorial. The code below uses a wide beam search: it grows only connected regions one neighboring qubit at a time, scores each partial region, and preserves the strongest 5,000 partial candidates at every size. This is deterministic, easy to audit, and avoids selecting disconnected collections of individually good qubits.


In [5]:
def score_region(nodes):
    nodes = frozenset(nodes)
    internal_edges = [(u, v) for u in nodes for v in adjacency[u] if u < v and v in nodes]
    if not internal_edges:
        return float("-inf"), 0, 0.0, 0.0
    mean_node = statistics.mean(node_quality[q] for q in nodes)
    mean_edge = statistics.mean(edge_quality[tuple(sorted((u, v)))] for u, v in internal_edges)
    compactness = len(internal_edges) / (len(nodes) - 1)
    score = 0.50 * mean_node + 0.40 * mean_edge + 0.10 * min(compactness / 1.4, 1.0)
    return score, len(internal_edges), mean_node, mean_edge

def best_connected_regions(size, beam_width=5000):
    beam = {frozenset([q]): node_quality[q] for q in by_qubit}
    for _ in range(1, size):
        expanded = {}
        for region in beam:
            frontier = set().union(*(adjacency[q] for q in region)) - set(region)
            for qubit in frontier:
                grown = region | {qubit}
                score, *_ = score_region(grown)
                if score > expanded.get(grown, float("-inf")):
                    expanded[grown] = score
        beam = dict(sorted(expanded.items(), key=lambda item: item[1], reverse=True)[:beam_width])
    return [(region, *score_region(region)) for region in sorted(beam, key=beam.get, reverse=True)]

top16 = best_connected_regions(16)
top32 = best_connected_regions(32)
region16, score16, edges16, node16, edge16 = top16[0]
region32, score32, edges32, node32, edge32 = top32[0]


## 5. Recommended 16-qubit region


In [7]:
def region_summary(region):
    nodes = sorted(region)
    internal = [(u, v) for u in region for v in adjacency[u] if u < v and v in region]
    return {
        "qubits": nodes,
        "internal_couplers": len(internal),
        "mean_readout_assignment_error": statistics.mean(float(by_qubit[q]["Readout assignment error"]) for q in nodes),
        "mean_sx_error": statistics.mean(float(by_qubit[q]["√x (sx) error"]) for q in nodes),
        "median_T1_us": statistics.median(float(by_qubit[q]["T1 (us)"]) for q in nodes),
        "median_T2_us": statistics.median(float(by_qubit[q]["T2 (us)"]) for q in nodes),
        "mean_internal_CZ_error": statistics.mean(raw_edge_error[tuple(sorted(edge))] for edge in internal),
    }

summary16 = region_summary(region16)
print("Recommended 16-qubit region:", summary16["qubits"])
for metric, value in summary16.items():
    if metric != "qubits":
        print(f"{metric}: {value:.6g}" if isinstance(value, float) else f"{metric}: {value}")
print(f"Composite score: {score16:.6f}; mean node-quality percentile: {node16:.4f}; mean coupler-quality percentile: {edge16:.4f}")


Recommended 16-qubit region: [97, 104, 105, 106, 107, 117, 122, 123, 124, 125, 126, 136, 140, 141, 142, 143]
internal_couplers: 15
mean_readout_assignment_error: 0.00828552
mean_sx_error: 0.000220996
median_T1_us: 254.31
median_T2_us: 224.513
mean_internal_CZ_error: 0.00134751
Composite score: 0.687347; mean node-quality percentile: 0.6062; mean coupler-quality percentile: 0.7820


**Rationale.** This is a connected 16-qubit region chosen by the stated joint node-and-coupler objective. It avoids qubits with missing required calibration and has favorable average readout, `sx`, and internal CZ metrics relative to the device snapshot. The internal coupler count reported above verifies its topology feasibility.


## 6. Recommended 32-qubit region


In [8]:
summary32 = region_summary(region32)
print("Recommended 32-qubit region:", summary32["qubits"])
for metric, value in summary32.items():
    if metric != "qubits":
        print(f"{metric}: {value:.6g}" if isinstance(value, float) else f"{metric}: {value}")
print(f"Composite score: {score32:.6f}; mean node-quality percentile: {node32:.4f}; mean coupler-quality percentile: {edge32:.4f}")


Recommended 32-qubit region: [59, 73, 74, 75, 79, 90, 91, 92, 93, 94, 97, 98, 103, 104, 105, 106, 107, 108, 109, 110, 111, 117, 122, 123, 124, 125, 126, 136, 140, 141, 142, 143]
internal_couplers: 31
mean_readout_assignment_error: 0.0108528
mean_sx_error: 0.00024097
median_T1_us: 263.157
median_T2_us: 211.044
mean_internal_CZ_error: 0.00154192
Composite score: 0.637390; mean node-quality percentile: 0.5804; mean coupler-quality percentile: 0.6894


**Rationale.** The 32-qubit recommendation optimizes the same transparent criterion at the larger connected scale. It is not simply the best 16-qubit set padded with nearby qubits: it is searched independently, because the quality/connectivity trade-off changes with region size.

## 7. Additional information

The recommendation is calibrated to the provided point-in-time metrics and is workload-agnostic. (1) repeated calibrations to measure drift and uncertainty; (2) two-qubit error by gate direction and simultaneous-operation context; (3) readout crosstalk and correlated-error measurements; (4) a representative circuit family, including two-qubit gate count, measurement pattern, and required logical connectivity; and (5) benchmark results such as mirror circuits or cycle benchmarking for each candidate region.


## 8.Region Selection Methodology

I selected the regions by checking the calibration quality and connectivity of the qubits. I preferred qubits with low readout error, low sx gate error, good T1 and T2 coherence times, and low CZ error between connected qubits. I also made sure that every qubit in a selected region is connected to the others through the hardware topology.

The recommended 16-qubit region is [97, 104, 105, 106, 107, 117, 122, 123, 124, 125, 126, 136, 140, 141, 142, 143]. This region was selected because its qubits are connected and have good overall calibration values, with reliable readout, single-qubit operations, and CZ connections.

The recommended 32-qubit region is [59, 73, 74, 75, 79, 90, 91, 92, 93, 94, 97, 98, 103, 104, 105, 106, 107, 108, 109, 110, 111, 117, 122, 123, 124, 125, 126, 136, 140, 141, 142, 143]. This region was selected because it is fully connected and gives the best balanced performance for a larger circuit.

Other regions were not selected because they had higher errors, lower coherence times, missing calibration data, or weaker CZ connections